In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
from tqdm import tqdm

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['nature', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()
coordinates = coordinates.read().result()

f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [77, 66, 67, 68, 69, 70, 71, 101, 102, 103, 106]
# mask_idx_list = [106] # visual processing
# mask_idx_list = [96, 97, 200] # eye movement control
# mask_idx_list = [65] # pretectum
mask_idx_list = [108] # torus longitudinalis [almost all neurons that end here come from pretectum and tectum neuropil]
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1


valid_coordinate_i = []
for i, c in enumerate(coordinates):
  if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
    valid_coordinate_i.append(i)
valid_coordinate_i = np.array(valid_coordinate_i)

In [ ]:
valid_coordinate_i.shape

Cluster selected traces by cutting them into fixed time-windows [0, T] and stacking

In [ ]:
traces_selected = traces[:, valid_coordinate_i]

In [ ]:
valid_coordinate_i_h = np.mean(traces_selected, 0) > 0.0
valid_coordinate_i_h_l = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] < 300)
valid_coordinate_i_h_r = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] > 300)

In [ ]:
# selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128]

In [ ]:
context = 4
T_stim_period = 140
selected_valid_coordinates_i = valid_coordinate_i[valid_coordinate_i_h]
len(selected_valid_coordinates_i)

In [ ]:
n_ix = 11
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_valid_coordinates_i[:n_ix])):
    ax = axs[i]
    ax.plot(traces[:500, neuron_ix], 'k')
    format_ax(ax)

In [ ]:
traces[:, selected_valid_coordinates_i].shape, selected_valid_coordinates_i.shape

In [ ]:
T_traces = np.concatenate([traces[T_stim_period*i:T_stim_period*(i+1), selected_valid_coordinates_i] for i in range(len(traces)//T_stim_period)], -1)

In [ ]:
# n_ix = 27
# fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
# axs = axs.flatten()

# for i in range(n_ix):
#     ax = axs[i]
#     ax.plot(T_traces[i][:T_stim_period*4], 'k')
#     format_ax(ax)

In [ ]:
T_traces.shape

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import numpy as np

T_traces_reshaped = T_traces.T
tsne = TSNE(n_components=2, random_state=0, perplexity=30)
T_traces_tsne = tsne.fit_transform(T_traces_reshaped)

In [ ]:
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
cluster_labels = kmeans.fit_predict(T_traces_tsne)

plt.figure(figsize=(2, 2), dpi=2000)
plt.rcParams.update({'font.family': 'Arial'})
colors = plt.cm.magma(np.linspace(0, 1, n_clusters))
for label in range(n_clusters):
    idx = cluster_labels == label
    plt.scatter(T_traces_tsne[idx, 0], T_traces_tsne[idx, 1], s=5, color=colors[label])
plt.title("t-SNE, torus longitudinalis", fontname="Arial")
plt.show()

In [ ]:
cluster_labels.shape, T_traces.shape

In [ ]:
tsne = TSNE(n_components=2, random_state=0, perplexity=20)
T_traces_tsne = tsne.fit_transform(traces[:, selected_valid_coordinates_i].T)

plt.figure()
plt.scatter(T_traces_tsne[:,0], T_traces_tsne[:,1], s=5)
plt.show()

In [ ]:
n_clusters = 4
n_to_show = 10

fig, axs = plt.subplots(n_clusters, n_to_show, figsize=(n_to_show, n_clusters), dpi=500)
if n_clusters == 1: axs = axs[None, :]
if n_to_show == 1: axs = axs[:, None]

for label in range(n_clusters):
    idx = np.where(cluster_labels == label)[0]
    show_n = min(n_to_show, len(idx))
    ax0 = axs[label, 0]
    ax0.text(-0.05, 0.5, f"Cluster {label}", transform=ax0.transAxes, fontsize=14,
             va='center', ha='right')
    if len(idx) > 0:
        chosen = np.random.choice(idx, show_n, replace=False)
        for j in range(show_n):
            ax = axs[label, j]
            t = T_traces[:, chosen[j]]
            ax.plot(t[:T_stim_period], 'k')
            format_ax(ax)
        for j in range(show_n, n_to_show):
            axs[label, j].axis("off")
    else:
        for j in range(n_to_show):
            axs[label, j].axis("off")
plt.tight_layout()
plt.show()